# ASP25/124 H-Bond Frequency Clustermap

**What this notebook does:**
- Reads per-simulation interaction CSV files (binary H-bond presence per frame)
- Filters columns to those involving **ASP25, ASP124** and nearby residues
  (residue IDs 23–28 and 123–128)
- Computes the **% frequency** of each H-bond in each simulation
- Keeps only bonds present in ≥ 25 % of frames in at least one simulation
- Produces a **seaborn clustermap** — rows are H-bonds (clustered), columns are
  simulations (kept in order)

**Sections:**
1. Imports
2. Configuration (paths, residue filter)
3. Helper functions: `find_asp_int()`, `analyze_asp_hbonds()`, `get_res_num()`
4. Compute frequency table
5. Filter and rename columns
6. Plot clustermap


In [ ]:
# 1. Imports
import os
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# 2. Configuration
ROOTDIR    = '/home/sdv/m1isdd/aperova/Documents/M1_STAGE/Data/interactions/'
# If running on Colab, replace with:
# ROOTDIR = '/content/drive/MyDrive/M1_STAGE/Data/interactions/'
OUTPUT_DIR = '/home/sdv/m1isdd/aperova/Documents/M1_STAGE/Manips/Figures/'

INTERACTION_FILES = [
    'res_V1.csv', 'res_V11.csv', 'res_V12.csv',
    'res_V7.csv', 'res_V8.csv', 'res_V21.csv'
]
SIMULATION_NAMES = ['V1', 'V11', 'V12', 'V7', 'V8', 'V21']

# Residue IDs to include in the H-bond search
RESID_LIST = ['23','24','25','26','27','28','123','124','125','126','128']

# Minimum frequency threshold (%) to retain a bond
MIN_FREQ = 25

os.chdir(ROOTDIR)

In [ ]:
# 3. Helper functions

def find_asp_int(interactions_file: str) -> None:
    """Print all H-bond columns involving ASP_25 that are ever non-zero."""
    df = pd.read_csv(interactions_file)
    for col in [c for c in df.columns if 'ASP_25' in c and 'hb' in c]:
        if df[col].sum() > 0:
            print(col)


def analyze_asp_hbonds(interactions_file: str) -> dict:
    """
    Compute the % frequency of H-bonds involving residues in RESID_LIST.

    Returns
    -------
    dict mapping column name → frequency (%)
    """
    df = pd.read_csv(interactions_file)
    n_frames = len(df)
    hb_cols = [c for c in df.columns if 'hb' in c and any(r in c for r in RESID_LIST)]
    return {col: (df[col] > 0).sum() / n_frames * 100 for col in hb_cols}


def get_res_num(interaction_string: str) -> int:
    """Extract the first residue number from a bond column name."""
    match = re.search(r'\d+', interaction_string)
    return int(match.group()) if match else 0

In [ ]:
# 4. Compute frequency table across all simulations
all_results = []
for sim_file in INTERACTION_FILES:
    data = analyze_asp_hbonds(sim_file)
    all_results.append(data)

freq_table = pd.DataFrame(all_results, index=SIMULATION_NAMES)
print(f'Frequency table shape: {freq_table.shape} (simulations × bonds)')
freq_table.head()

In [ ]:
# 5. Filter: keep bonds present in >= MIN_FREQ % in at least one simulation
filtered_cols = freq_table.columns[(freq_table > MIN_FREQ).any()]
freq_filtered = freq_table[filtered_cols].copy()
print(f'Bonds retained after {MIN_FREQ}% filter: {len(filtered_cols)}')

# Build human-readable column labels
descriptive_labels = []
for bond in freq_filtered.columns:
    parts = bond.split('_')
    try:
        res1 = f"{parts[1]:<3}{parts[2]:>3}_{parts[3]:<3}"
        res2 = f"{parts[4]:<3}{parts[5]:>3}_{parts[6]:<3}"
        descriptive_labels.append(f"{res2} - {res1}")
    except IndexError:
        descriptive_labels.append(bond)  # fallback to raw name

freq_filtered = freq_filtered.rename(
    columns=dict(zip(freq_filtered.columns, descriptive_labels))
)

# Sort columns by first residue number
sorted_cols = sorted(freq_filtered.columns, key=get_res_num)
df_sorted   = freq_filtered[sorted_cols]
df_sorted.head()

In [ ]:
# 6. Plot clustermap (rows = H-bonds clustered; columns = simulations in order)
g = sns.clustermap(
    df_sorted.T,
    cmap='Blues',
    col_cluster=False,   # preserve simulation order on x-axis
    row_cluster=True,    # group similar H-bonds together on y-axis
    figsize=(6, max(10, len(df_sorted.columns) * 0.35)),
    cbar_kws={'label': 'H-bond Frequency (%)'}
)

g.fig.suptitle('Hydrogen Bond Frequency — ASP25/ASP124 region', y=1.01, fontsize=14)
g.ax_heatmap.set_xlabel('Simulation', fontsize=10)
g.ax_heatmap.set_ylabel('Interaction Type', fontsize=10)
g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(), family='monospace', rotation=45, ha='right', fontsize=9)
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(), family='monospace', fontsize=9)

plt.tight_layout()
plt.show()
g.fig.savefig(os.path.join(OUTPUT_DIR, 'interactions_heatmap_clustermap.png'),
              bbox_inches='tight', dpi=300)